In [1]:
from google.colab import files
uploaded = files.upload()

Saving ecommerce_customer_churn_dataset.csv to ecommerce_customer_churn_dataset.csv


In [3]:
import pandas as pd

df = pd.read_csv("ecommerce_customer_churn_dataset.csv")

df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Email_Open_Rate,Customer_Service_Calls,Product_Reviews_Written,Social_Media_Engagement_Score,Mobile_App_Usage,Payment_Method_Diversity,Lifetime_Value,Credit_Balance,Churned,Signup_Quarter
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,17.9,9.0,4.0,16.3,20.8,1.0,953.33,2278.0,0,Q1
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,42.8,7.0,3.0,NaN,23.3,3.0,1067.47,3028.0,0,Q4
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0.0,4.0,1.0,NaN,8.8,NaN,1289.75,2317.0,0,Q4
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,41.4,2.0,5.0,85.9,31.0,3.0,2340.92,2674.0,0,Q1
4,35.0,Male,India,Delhi,3.1,29.0,51.4,NaN,19.1,9.0,...,37.9,1.0,11.0,83.0,50.4,4.0,3041.29,5354.0,0,Q4


In [4]:
import numpy as np

df["engagement_level"] = pd.qcut(
    df["Login_Frequency"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [5]:
summary = df.groupby("engagement_level")["Churned"].agg(
    total_users="count",
    churned_users="sum"
)

summary

/tmp/ipykernel_153/3608127566.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = df.groupby("engagement_level")["Churned"].agg(


,total_users,churned_users
engagement_level,,
Low,19115,7626
Medium,16071,3878
High,14814,2946


In [6]:
low = summary.loc["Low"]
medium = summary.loc["Medium"]
high = summary.loc["High"]

control_total = low["total_users"]
control_churn = low["churned_users"]

variant_total = medium["total_users"] + high["total_users"]
variant_churn = medium["churned_users"] + high["churned_users"]

control_total, control_churn, variant_total, variant_churn

(np.int64(19115), np.int64(7626), np.int64(30885), np.int64(6824))

In [7]:
from statsmodels.stats.proportion import proportions_ztest

count = [control_churn, variant_churn]
nobs = [control_total, variant_total]

z_stat, p_value = proportions_ztest(count, nobs)

z_stat, p_value

(np.float64(42.67018069925312), np.float64(0.0))

In [8]:
control_rate = control_churn / control_total
variant_rate = variant_churn / variant_total

uplift = (control_rate - variant_rate) / control_rate * 100

control_rate, variant_rate, uplift

(np.float64(0.3989537012817159),
 np.float64(0.22094868058928283),
 np.float64(44.61796447070363))